In [1]:
import gradio as gr
import nltk
from nltk.probability import FreqDist
from nltk.corpus import stopwords
import string
import heapq

# --- NLTK Setup ---
nltk.download('punkt')
nltk.download('stopwords')

def process_corpus(input_text, file_obj, target_word):
    raw_text = ""
    if file_obj is not None:
        try:
            with open(file_obj.name, 'r', encoding='utf-8') as f:
                raw_text = f.read()
        except Exception as e:
            return f"Error: {str(e)}", "", ""
    elif input_text.strip():
        raw_text = input_text
    else:
        return "စာသားထည့်ပါ သို့မဟုတ် .txt ဖိုင်တင်ပေးပါ။", "", ""

    tokens = nltk.word_tokenize(raw_text)
    sentences = nltk.sent_tokenize(raw_text)
    stop_words = set(stopwords.words('english'))
    words = [w.lower() for w in tokens if w.lower() not in stop_words and w not in string.punctuation]
    word_frequencies = FreqDist(words)
    
    sent_scores = {}
    for sent in sentences:
        for word in nltk.word_tokenize(sent.lower()):
            if word in word_frequencies:
                if sent not in sent_scores:
                    sent_scores[sent] = word_frequencies[word]
                else:
                    sent_scores[sent] += word_frequencies[word]
    
    summary_sentences = heapq.nlargest(5, sent_scores, key=sent_scores.get)
    summary_res = " ".join(summary_sentences) if summary_sentences else "အနှစ်ချုပ်ရန် လုံလောက်သော စာသားမရှိပါ။"

    concordance_list = []
    if target_word.strip():
        search_term = target_word.strip().lower()
        for i, token in enumerate(tokens):
            if token.lower() == search_term:
                start = max(0, i - 5)
                end = min(len(tokens), i + 6)
                context = tokens[start:end]
                concordance_list.append("... " + " ".join(context) + " ...")
        concordance_res = "\n\n".join(concordance_list[:15]) if concordance_list else "ရှာဖွေနေသော စကားလုံး မတွေ့ရှိပါ။"
    else:
        concordance_res = "ရှာဖွေရန် စကားလုံးကို ရိုက်ထည့်ပါ။"

    fdist = FreqDist(words)
    all_words = fdist.most_common() 
    stats_res = f"စုစုပေါင်း စကားလုံး: {len(tokens)}\nထူးခြားသော စကားလုံး: {len(fdist)}\n" + "-"*30 + "\n"
    for word, count in all_words:
        stats_res += f"{word}: {count}\n"

    return summary_res, concordance_res, stats_res

# --- Custom Styling (Font Size များ တိုးမြှင့်ထားသည်) ---
custom_css = """
.header-banner {
    background: linear-gradient(90deg, #8E2DE2 0%, #4A00E0 100%);
    padding: 60px 20px;
    border-radius: 25px;
    text-align: center;
    color: white !important;
    margin-bottom: 30px;
}
.header-banner h1 { color: white !important; font-size: 42px !important; font-weight: 800; margin-bottom: 15px; }
.header-banner p { color: #dcdcdc !important; font-size: 22px !important; }

/* Card CSS & Animations */
.feature-card {
    background: white; padding: 40px 20px; border-radius: 20px; text-align: center;
    box-shadow: 0 4px 15px rgba(0,0,0,0.05); border: 1px solid #f0f0f0; height: 100%;
    transition: all 0.3s cubic-bezier(.25,.8,.25,1);
}
.feature-card:hover { transform: translateY(-10px); box-shadow: 0 14px 28px rgba(74, 0, 224, 0.15); border-color: #8E2DE2; }

/* Font Sizes inside Cards */
.feature-card h3 { font-size: 28px !important; font-weight: 700 !important; margin: 15px 0 !important; }
.feature-card p { font-size: 19px !important; line-height: 1.5; color: #555 !important; }

/* Button Styling & Animation */
.get-started-btn {
    background-color: #4CAF50 !important; color: white !important;
    border-radius: 12px !important; font-size: 24px !important; font-weight: 600 !important;
    padding: 20px 40px !important; transition: all 0.2s ease-in-out !important;
}
.get-started-btn:hover { transform: scale(1.05); filter: brightness(1.1); animation: wobble 0.5s ease; }

@keyframes wobble {
  0% { transform: scale(1.05) rotate(0deg); }
  25% { transform: scale(1.05) rotate(2deg); }
  50% { transform: scale(1.05) rotate(-2deg); }
  75% { transform: scale(1.05) rotate(1deg); }
  100% { transform: scale(1.05) rotate(0deg); }
}

/* About Us Section Font Size */
.about-content { font-size: 15px !important; line-height: 1.8 !important; }
.about-content b, .about-content strong { font-size: 17px !important; color: black; }
"""

with gr.Blocks() as demo:
    with gr.Tabs() as main_tabs:
        
        with gr.Tab("🏠 HOME", id=0):
            with gr.Column(elem_classes="header-banner"):
                gr.Markdown("# 🧠 Text Corpus Exploration System")
                gr.Markdown("Analyze your text with the power of AI & NLP")
            
            with gr.Row():
                with gr.Column(elem_classes="feature-card"):
                    gr.Markdown("✨\n### AI Summary\nGet the core points instantly.")
                with gr.Column(elem_classes="feature-card"):
                    gr.Markdown("📊\n### Statistics\nWord counts & frequencies.")
                with gr.Column(elem_classes="feature-card"):
                    gr.Markdown("🔍\n### Context\nSearch word usage patterns.")
            
            gr.HTML("<br><br>")
            with gr.Row():
                gr.Markdown(" ")
                start_btn = gr.Button("Get Started Now 🚀", elem_classes="get-started-btn", scale=1)
                gr.Markdown(" ")

        with gr.Tab("🔍 TEXT EXPLORER", id=1):
            with gr.Row():
                with gr.Column(scale=1):
                    file_input = gr.File(label=".txt ဖိုင်တင်ရန်", file_types=[".txt"])
                    text_input = gr.Textbox(label="စာသားထည့်ရန်", lines=10, placeholder="Paste text here...", elem_id="large_txt")
                    target_word = gr.Textbox(label="ရှာဖွေလိုသော စကားလုံး", placeholder="e.g., intelligence")
                    with gr.Row():
                        btn = gr.Button("စတင်စစ်ဆေးမည်", variant="primary")
                        clear_btn = gr.Button("ဖျက်မည်")
                with gr.Column(scale=1):
                    out_summary = gr.Textbox(label="✨ AI အနှစ်ချုပ်", lines=6)
                    out_concordance = gr.Textbox(label="🔍 စကားလုံးအသုံးပြုပုံ (Concordance)", lines=6)
                    out_stats = gr.Textbox(label="📊 စာရင်းအင်းအချက်အလက်များ", lines=8)

        with gr.Tab("ℹ️ ABOUT US", id=2):
            with gr.Column(elem_classes="about-content"):
                gr.Markdown("""
                ### ℹ️ Project Overview
                * *🧠 **Text Corpus Exploration System*** သည် လူသားတို့၏ ဘာသာစကားကို AI နည်းပညာများဖြင့် စိတ်ဖြာလေ့လာရန် ဖန်တီးထားခြင်း ဖြစ်ပါသည်။

                ### 🛠️ Core AI Technologies
                * **AI Summarization:** စာသားတစ်ခုအတွင်းရှိ အရေးကြီးဆုံး ဝါကျများကို Frequency Score များတွက်ချက်၍ အနှစ်ချုပ်ပေးခြင်း။
                * **Tokenization:** စာသားများကို အစိတ်အပိုင်းငယ်များ (tokens) အဖြစ် ခွဲခြမ်းစိတ်ဖြာခြင်း။
                * **Contextual Search:** စကားလုံးတစ်လုံးချင်းစီ၏ အသုံးပြုပုံကို ပတ်ဝန်းကျင်ဝါကျများနှင့်တကွ ရှာဖွေပေးခြင်း။
                * **Stop-word Filtering:** အဓိပ္ပာယ်ဖော်ဆောင်မှုနည်းသော စကားလုံးများကို ဖယ်ထုတ်၍ အဓိကအချက်အလက်များကိုသာ ဦးစားပေးခြင်း။

                ### 👤 Developer Information
                * **အမည်:** မသိမ့်နန္ဒာထက် (PaKaPaTa-002127)
                * **အတန်း:** 5CS(Section-A)
                """)

    def go_to_explorer():
        return gr.update(selected=1)

    start_btn.click(fn=go_to_explorer, inputs=None, outputs=main_tabs)
    btn.click(fn=process_corpus, inputs=[text_input, file_input, target_word], outputs=[out_summary, out_concordance, out_stats])
    clear_btn.click(fn=lambda: (None, "", "", "", "", ""), outputs=[file_input, text_input, target_word, out_summary, out_concordance, out_stats])

if __name__ == "__main__":
    demo.launch(css=custom_css, theme=gr.themes.Soft(primary_hue="purple"))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
